Basically try to make an automation search bot. its my personal practice on python

In [ ]:
import pandas as pd
 = pd.read_csv("movies.csv")

In [ ]:
movies

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,ToyStory1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji1995
2,3,Grumpier Old Men (1995),Comedy|Romance,GrumpierOldMen1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,WaitingtoExhale1995
4,5,Father of the Bride Part II (1995),Comedy,FatheroftheBridePartII1995
...,...,...,...,...
62418,209157,We (2018),Drama,We2018
62419,209159,Window of the Soul (2001),Documentary,WindowoftheSoul2001
62420,209163,Bad Poems (2018),Comedy|Drama,BadPoems2018
62421,209169,A Girl Thing (2001),(no genres listed),AGirlThing2001


In [ ]:
import re

def clean_title(title):
  return re.sub("[^a-zA-Z0-9]","",title)


In [ ]:
movies["clean_title"] = movies["title"].apply(clean_title)

In [ ]:
movies

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,ToyStory1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji1995
2,3,Grumpier Old Men (1995),Comedy|Romance,GrumpierOldMen1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,WaitingtoExhale1995
4,5,Father of the Bride Part II (1995),Comedy,FatheroftheBridePartII1995
...,...,...,...,...
62418,209157,We (2018),Drama,We2018
62419,209159,Window of the Soul (2001),Documentary,WindowoftheSoul2001
62420,209163,Bad Poems (2018),Comedy|Drama,BadPoems2018
62421,209169,A Girl Thing (2001),(no genres listed),AGirlThing2001


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1,2))

tfidf = vectorizer.fit_transform(movies["clean_title"])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search(title):
  title = "toy story"
  title = clean_title(title)
  query_vec = vectorizer.transform([title])
  similarity = cosine_similarity(query_vec, tfidf).flatten()
  indices = np.argpartition(similarity,-5)[-5:]
  results = movies.iloc[indices][::-1]
  return results




In [ ]:
import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
    value="Toy Story",
    Description = "Movie Title",
    disabled=False
)
movie_list = widgets.Output()

def on_type(data):
  with movie_list:
    movie_list.clear_output()
    title = data["new"]
    if len(title)>5:
      display(search(title))


movie_input.observe(on_type, names='value')

display(movie_input, movie_list)

Text(value='Toy Story')

Output()

In [ ]:
ratings = pd.read_csv("ratings.csv")

In [ ]:
ratings

,userId,movieId,rating,timestamp
0,1,296,5.0,1.147880e+09
1,1,306,3.5,1.147869e+09
2,1,307,5.0,1.147869e+09
3,1,665,5.0,1.147879e+09
4,1,899,3.5,1.147869e+09
...,...,...,...,...
12064466,78253,110,4.5,1.087548e+09
12064467,78253,111,4.5,1.087612e+09
12064468,78253,145,3.0,1.087547e+09
12064469,78253,150,3.0,1.087548e+09


In [ ]:
ratings.dtypes

userId         int64
movieId        int64
rating       float64
timestamp    float64
dtype: object

In [ ]:
movie_id= 1

In [ ]:
similar_users = ratings [(ratings["movieId"]==movie_id)&(ratings["rating"]>=4)] ["userId"].unique()

In [ ]:
similar_users

array([    3,     5,     8, ..., 78240, 78241, 78253])

In [ ]:
similar_users_recs = ratings [(ratings["userId"].isin(similar_users))&(ratings["rating"]>=4)]["movieId"]

In [ ]:
similar_users_recs

254           1
255          29
256          32
257          50
258         111
           ... 
12064459     16
12064462     47
12064463     50
12064466    110
12064467    111
Name: movieId, Length: 2455141, dtype: int64

In [ ]:
similar_users_recs = similar_users_recs.value_counts() / len(similar_users)
similar_users_recs = similar_users_recs[similar_users_recs >.10]

In [ ]:
similar_users_recs

movieId
1       1.000000
318     0.548178
260     0.524612
356     0.518941
296     0.492732
          ...   
11      0.102412
2761    0.101145
3948    0.100595
1358    0.100264
1997    0.100099
Name: count, Length: 272, dtype: float64

In [ ]:
all_users = ratings[(ratings["movieId"].isin(similar_users_recs.index))&(ratings["rating"]>4)]

In [ ]:
all_users_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

In [ ]:
all_users_recs

movieId
318     0.331166
296     0.274591
2571    0.235866
356     0.226528
593     0.217203
          ...   
3175    0.019556
3948    0.019502
474     0.018823
440     0.016452
2       0.016452
Name: count, Length: 272, dtype: float64

In [ ]:
rec_percentages = pd.concat ([similar_users_recs, all_users_recs], axis=1)
rec_percentages.columns = ["similar","all"]

In [ ]:
rec_percentages

,similar,all
movieId,,
1,1.000000,0.121370
318,0.548178,0.331166
260,0.524612,0.213353
356,0.518941,0.226528
296,0.492732,0.274591
...,...,...
11,0.102412,0.021154
2761,0.101145,0.019809
3948,0.100595,0.019502


In [ ]:
rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]

In [ ]:
rec_percentages = rec_percentages.sort_values("score", ascending=False)

In [ ]:
rec_percentages

,similar,all,score
movieId,,,
1,1.000000,0.121370,8.239271
2355,0.193811,0.024298,7.976438
648,0.189351,0.027815,6.807582
2,0.105715,0.016452,6.425774
440,0.104834,0.016452,6.372226
...,...,...,...
858,0.351558,0.202643,1.734865
2959,0.352164,0.208598,1.688246
109487,0.119700,0.072241,1.656957


In [ ]:
rec_percentages.head(10).merge(movies,left_index=True, right_on= "movieId")

,similar,all,score,movieId,title,genres,clean_title
0,1.000000,0.121370,8.239271,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,ToyStory1995
2264,0.193811,0.024298,7.976438,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,BugsLifeA1998
637,0.189351,0.027815,6.807582,648,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller,MissionImpossible1996
1,0.105715,0.016452,6.425774,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji1995
435,0.104834,0.016452,6.372226,440,Dave (1993),Comedy|Romance,Dave1993
3021,0.330966,0.051993,6.365600,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,ToyStory21999
584,0.198161,0.031931,6.205903,592,Batman (1989),Action|Crime|Thriller,Batman1989
3650,0.129501,0.020954,6.180161,3751,Chicken Run (2000),Animation|Children|Comedy,ChickenRun2000
2705,0.153177,0.025164,6.087183,2797,Big (1988),Comedy|Drama|Fantasy|Romance,Big1988
721,0.128015,0.021474,5.961411,736,Twister (1996),Action|Adventure|Romance|Thriller,Twister1996


In [ ]:
def find_similar_movies(movie_id):
  similar_users = ratings [(ratings["movieId"]==movie_id)&(ratings["rating"]>=4)] ["userId"].unique()
  similar_users_recs = ratings [(ratings["userId"].isin(similar_users))&(ratings["rating"]>=4)]["movieId"]

  similar_users_recs = similar_users_recs.value_counts() / len(similar_users)
  similar_users_recs = similar_users_recs[similar_users_recs >.10]

  all_users = ratings[(ratings["movieId"].isin(similar_users_recs.index))&(ratings["rating"]>4)]
  all_users_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

  rec_percentages = pd.concat ([similar_users_recs, all_users_recs], axis=1)
  rec_percentages.columns = ["similar","all"]

  rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]

  rec_percentages = rec_percentages.sort_values("score", ascending=False)
  return rec_percentages.head(10).merge(movies,left_index=True, right_on= "movieId")[["score","title","genres"]]



In [ ]:
movie_name_input = widgets.Text(
    value="Toy Story",
    Description = "Movie Title",
    disabled=False
)

recommendation_list = widgets.Output()

def on_type(data):
  with recommendation_list:
    recommendation_list.clear_output()
    title = data["new"]
    if len(title)>5:
      result = search(title)
      movie_id = result.iloc[0]["movieId"]
      display(find_similar_movies(movie_id))

movie_name_input.observe(on_type, names='value')

display(movie_name_input, recommendation_list)




Text(value='Toy Story')

Output()